# Description

Generates manubot tables for PhenomeXcan and eMERGE associations given an LV name (which is the only parameter that needs to be specified in the Settings section below).

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
from pathlib import Path

import pandas as pd

from entity import Trait
import conf

# Settings

In [3]:
LV_NAME = "LV24"

In [4]:
# assert (
#     conf.MANUSCRIPT["BASE_DIR"] is not None
# ), "The manuscript directory was not configured"

# OUTPUT_FILE_PATH = conf.MANUSCRIPT["CONTENT_DIR"] / "50.00.supplementary_material.md"
# display(OUTPUT_FILE_PATH)
# assert OUTPUT_FILE_PATH.exists()

In [5]:
# result_set is either phenomexcan or emerge
# LV_FILE_MARK_TEMPLATE = "<!-- {lv}:{result_set}_traits_assocs:{position} -->"

In [6]:
# TABLE_CAPTION = "Table: Significant trait associations of {lv_name} in {result_set_name}. {table_id}"

In [7]:
# TABLE_CAPTION_ID = "#tbl:sup:{result_set}_assocs:{lv_name_lower_case}"

In [8]:
RESULT_SET_NAMES = {
    "phenomexcan": "PhenomeXcan",
    "emerge": "eMERGE",
}

# Load data

## PhenomeXcan LV-trait associations

In [9]:
input_filepath = Path(conf.RESULTS["GLS"] / "gls-summary-phenomexcan.pkl.gz")
display(input_filepath)

PosixPath('/opt/data/results/gls/gls-summary-phenomexcan.pkl.gz')

In [11]:
phenomexcan_lv_trait_assocs = pd.read_pickle(input_filepath)

In [12]:
phenomexcan_lv_trait_assocs.shape

(4037817, 5)

In [13]:
phenomexcan_lv_trait_assocs.head()

,phenotype,phenotype_desc,lv,pvalue,fdr
0,AB1_OTHER_VIRAL,Other viral diseases,LV736,0.004725,0.504339
1,AB1_OTHER_VIRAL,Other viral diseases,LV320,0.004848,0.508291
2,AB1_OTHER_VIRAL,Other viral diseases,LV366,0.005306,0.523691
3,AB1_OTHER_VIRAL,Other viral diseases,LV964,0.006106,0.548143
4,AB1_OTHER_VIRAL,Other viral diseases,LV92,0.006565,0.560048


In [64]:
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    _tmp = phenomexcan_lv_trait_assocs[
        (phenomexcan_lv_trait_assocs["lv"] == "LV24")
        & (phenomexcan_lv_trait_assocs["pvalue"] < 0.01)
    ].sort_values("pvalue")
    
    display(_tmp)

,phenotype,phenotype_desc,lv,pvalue,fdr
1367986,4101_raw,Heel broadband ultrasound attenuation (left),LV24,1.850282e-21,1.464922e-16
1064977,4120_raw,Heel broadband ultrasound attenuation (right),LV24,2.169228e-21,1.684413e-16
2039146,4124_raw,Heel bone mineral density (BMD) (right),LV24,4.254136e-12,4.580646e-08
463894,4123_raw,"Heel quantitative ultrasound index (QUI), direct entry (right)",LV24,5.295114e-12,5.557142e-08
3587749,4125_raw,"Heel bone mineral density (BMD) T-score, automated (right)",LV24,5.298654e-12,5.557142e-08
4018083,MAGNETIC_CH2.DB.ratio,CH2DB NMR,LV24,8.883343e-11,7.441766e-07
790591,4105_raw,Heel bone mineral density (BMD) (left),LV24,9.692989e-11,8.036656e-07
118444,4104_raw,"Heel quantitative ultrasound index (QUI), direct entry (left)",LV24,1.357855e-10,1.082856e-06
2049016,4106_raw,"Heel bone mineral density (BMD) T-score, automated (left)",LV24,1.359666e-10,1.082856e-06
1587100,3148_raw,Heel bone mineral density (BMD),LV24,1.296695e-09,8.245383e-06


## eMERGE LV-trait associations

In [14]:
input_filepath = Path(conf.RESULTS["GLS"] / "gls-summary-emerge.pkl.gz")
display(input_filepath)

PosixPath('/opt/data/results/gls/gls-summary-emerge.pkl.gz')

In [15]:
emerge_lv_trait_assocs = pd.read_pickle(input_filepath)

In [16]:
emerge_lv_trait_assocs.shape

(304983, 5)

In [17]:
emerge_lv_trait_assocs.head()

,phenotype,phenotype_desc,lv,pvalue,fdr
0,EUR_440.2,Atherosclerosis of the extremities,LV472,1.033637e-07,0.000658
1,EUR_440.2,Atherosclerosis of the extremities,LV182,3.710244e-07,0.001432
2,EUR_440.2,Atherosclerosis of the extremities,LV348,7.379936e-07,0.002558
3,EUR_440.2,Atherosclerosis of the extremities,LV504,1.534424e-06,0.004500
4,EUR_440.2,Atherosclerosis of the extremities,LV445,2.912525e-06,0.007402


In [65]:
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    _tmp = emerge_lv_trait_assocs[
        (emerge_lv_trait_assocs["lv"] == "LV24")
        & (emerge_lv_trait_assocs["pvalue"] < 0.01)
    ].sort_values("pvalue")
    
    display(_tmp)

,phenotype,phenotype_desc,lv,pvalue,fdr
53301,EUR_743.1,Osteoporosis,LV24,0.001955,0.392564
76990,EUR_365,Glaucoma,LV24,0.002250,0.411932
35536,EUR_687.1,Rash and other nonspecific skin eruption,LV24,0.003306,0.467492


## eMERGE traits info

In [18]:
input_filepath = conf.EMERGE["DESC_FILE_WITH_SAMPLE_SIZE"]
display(input_filepath)

PosixPath('/opt/data/data/emerge/eMERGE_III_PMBB_GSA_v2_2020_phecode_AFR_EUR_cc50_counts_w_dictionary.txt')

In [20]:
emerge_traits_info = pd.read_csv(
    input_filepath,
    sep="\t",
    dtype={"phecode": str},
    usecols=[
        "phecode",
        "phenotype",
        "category",
        "eMERGE_III_EUR_case",
        "eMERGE_III_EUR_control",
    ],
)

In [21]:
emerge_traits_info["phecode"] = emerge_traits_info["phecode"].apply(
    lambda x: f"EUR_{x}"
)

In [22]:
emerge_traits_info = emerge_traits_info.set_index("phecode").sort_index()

In [23]:
emerge_traits_info = emerge_traits_info.rename(
    columns={
        "eMERGE_III_EUR_case": "eur_n_cases",
        "eMERGE_III_EUR_control": "eur_n_controls",
    }
)

In [24]:
emerge_traits_info.shape

(309, 4)

In [25]:
emerge_traits_info.head()

,eur_n_cases,eur_n_controls,phenotype,category
phecode,,,,
EUR_008,1639,57495,Intestinal infection,infectious diseases
EUR_008.5,1024,57495,Bacterial enteritis,infectious diseases
EUR_008.52,893,57495,Intestinal infection due to C. difficile,infectious diseases
EUR_038,3172,50610,Septicemia,infectious diseases
EUR_038.3,1361,50610,Bacteremia,infectious diseases


In [26]:
assert emerge_traits_info.index.is_unique

# Trait associations

## PhenomeXcan

In [27]:
from traits import SHORT_TRAIT_NAMES

In [28]:
result_set = "phenomexcan"

In [29]:
def get_trait_objs(phenotype_full_code):
    if Trait.is_efo_label(phenotype_full_code):
        traits = Trait.get_traits_from_efo(phenotype_full_code)
    else:
        traits = [Trait.get_trait(code=phenotype_full_code)]

    # sort by sample size
    return sorted(traits, key=lambda x: x.n_cases / x.n, reverse=True)


def get_trait_description(phenotype_full_code):
    traits = get_trait_objs(phenotype_full_code)

    desc = traits[0].description
    if desc in SHORT_TRAIT_NAMES:
        return SHORT_TRAIT_NAMES[desc]

    return desc


def get_trait_n(phenotype_full_code):
    traits = get_trait_objs(phenotype_full_code)

    return traits[0].n


def get_trait_n_cases(phenotype_full_code):
    traits = get_trait_objs(phenotype_full_code)

    return traits[0].n_cases


def num_to_int_str(num):
    if pd.isnull(num):
        return ""

    return f"{num:,.0f}"


def get_part_clust(row):
    return f"{row.part_k} / {row.cluster_id}"

In [30]:
lv_assocs = phenomexcan_lv_trait_assocs[
    (phenomexcan_lv_trait_assocs["lv"] == LV_NAME)
    & (phenomexcan_lv_trait_assocs["fdr"] < 0.05)
].sort_values("fdr")

In [31]:
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(lv_assocs)

,phenotype,phenotype_desc,lv,pvalue,fdr
1367986,4101_raw,Heel broadband ultrasound attenuation (left),LV24,1.850282e-21,1.464922e-16
1064977,4120_raw,Heel broadband ultrasound attenuation (right),LV24,2.169228e-21,1.684413e-16
2039146,4124_raw,Heel bone mineral density (BMD) (right),LV24,4.254136e-12,4.580646e-08
463894,4123_raw,"Heel quantitative ultrasound index (QUI), direct entry (right)",LV24,5.295114e-12,5.557142e-08
3587749,4125_raw,"Heel bone mineral density (BMD) T-score, automated (right)",LV24,5.298654e-12,5.557142e-08
4018083,MAGNETIC_CH2.DB.ratio,CH2DB NMR,LV24,8.883343e-11,7.441766e-07
790591,4105_raw,Heel bone mineral density (BMD) (left),LV24,9.692989e-11,8.036656e-07
118444,4104_raw,"Heel quantitative ultrasound index (QUI), direct entry (left)",LV24,1.357855e-10,1.082856e-06
2049016,4106_raw,"Heel bone mineral density (BMD) T-score, automated (left)",LV24,1.359666e-10,1.082856e-06
1587100,3148_raw,Heel bone mineral density (BMD),LV24,1.296695e-09,8.245383e-06


In [32]:
lv_assocs = lv_assocs.assign(
    phenotype_desc=lv_assocs["phenotype"].apply(get_trait_description)
)

In [33]:
lv_assocs = lv_assocs.assign(n=lv_assocs["phenotype"].apply(get_trait_n))

In [34]:
lv_assocs = lv_assocs.assign(n_cases=lv_assocs["phenotype"].apply(get_trait_n_cases))

In [35]:
# lv_assocs = lv_assocs.assign(coef=lv_assocs["coef"].apply(lambda x: f"{x:.3f}"))

In [36]:
lv_assocs = lv_assocs.assign(
    fdr=lv_assocs["fdr"].apply(lambda x: f"{x:.2e}".replace("-", "&#8209;"))
)

In [37]:
lv_assocs = lv_assocs.assign(n=lv_assocs["n"].apply(num_to_int_str))

In [38]:
lv_assocs = lv_assocs.assign(n_cases=lv_assocs["n_cases"].apply(num_to_int_str))

In [39]:
# lv_assocs = lv_assocs.assign(part_clust="")  # lv_assocs.apply(get_part_clust, axis=1))

In [40]:
lv_assocs = lv_assocs.drop(columns=["phenotype"])

In [41]:
lv_assocs.shape

(18, 6)

In [42]:
lv_assocs = lv_assocs[["phenotype_desc", "n", "n_cases", "fdr"]]

In [43]:
lv_assocs = lv_assocs.rename(
    columns={
        "part_clust": "Partition / cluster",
        "lv": "Latent variable (LV)",
        #         "coef": r"$\beta$",
        "fdr": "FDR",
        "phenotype_desc": "Trait description",
        "n": "Sample size",
        "n_cases": "Cases",
    }
)

In [44]:
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(lv_assocs)

,Trait description,Sample size,Cases,FDR
1367986,Heel broadband ultrasound attenuation (left),"114,625",,1.46e&#8209;16
1064977,Heel broadband ultrasound attenuation (right),"114,609",,1.68e&#8209;16
2039146,Heel bone mineral density (BMD) (right),"114,552",,4.58e&#8209;08
463894,"Heel quantitative ultrasound index, direct entry (right)","114,614",,5.56e&#8209;08
3587749,"Heel bone mineral density T-score, automated (right)","114,614",,5.56e&#8209;08
4018083,CH2DB NMR,"24,154",,7.44e&#8209;07
790591,Heel bone mineral density (BMD) (left),"114,561",,8.04e&#8209;07
118444,"Heel quantitative ultrasound index (QUI), direct entry (left)","114,630",,1.08e&#8209;06
2049016,"Heel bone mineral density (BMD) T-score, automated (left)","114,630",,1.08e&#8209;06
1587100,Heel bone mineral density (BMD),"206,496",,8.25e&#8209;06


## eMERGE

In [ ]:
epval = 0.005

In [46]:
result_set = "emerge"

In [47]:
lv_assocs = emerge_lv_trait_assocs[
    (emerge_lv_trait_assocs["lv"] == LV_NAME) & (emerge_lv_trait_assocs["fdr"] < 0.05)
].sort_values("fdr")

In [48]:
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(lv_assocs)

,phenotype,phenotype_desc,lv,pvalue,fdr


In [49]:
lv_assocs = lv_assocs.assign(
    phenotype_desc=lv_assocs["phenotype"].apply(
        lambda x: emerge_traits_info.loc[x, "phenotype"]
    )
)

In [50]:
lv_assocs = lv_assocs.assign(
    n=lv_assocs["phenotype"].apply(
        lambda x: emerge_traits_info.loc[x, ["eur_n_cases", "eur_n_controls"]].sum()
    )
)

In [51]:
lv_assocs = lv_assocs.assign(
    n_cases=lv_assocs["phenotype"].apply(
        lambda x: emerge_traits_info.loc[x, "eur_n_cases"]
    )
)

In [52]:
lv_assocs["phenotype"] = lv_assocs["phenotype"].apply(lambda x: x.split("EUR_")[1])

In [53]:
# lv_assocs = lv_assocs.assign(coef=lv_assocs["coef"].apply(lambda x: f"{x:.3f}"))

In [54]:
lv_assocs = lv_assocs.assign(
    fdr=lv_assocs["fdr"].apply(lambda x: f"{x:.2e}".replace("-", "&#8209;"))
)

In [55]:
lv_assocs = lv_assocs.assign(n=lv_assocs["n"].apply(num_to_int_str))

In [56]:
lv_assocs = lv_assocs.assign(n_cases=lv_assocs["n_cases"].apply(num_to_int_str))

In [57]:
lv_assocs = lv_assocs.rename(columns={"phenotype": "phecode"})

In [58]:
lv_assocs.shape

(0, 7)

In [59]:
lv_assocs = lv_assocs[["phecode", "phenotype_desc", "n", "n_cases", "fdr"]]

In [60]:
lv_assocs = lv_assocs.rename(
    columns={
        "lv": "Latent variable (LV)",
        #         "coef": r"$\beta$",
        "fdr": "FDR",
        "phecode": "Phecode",
        "phenotype_desc": "Trait description",
        "n": "Sample size",
        "n_cases": "Cases",
    }
)

In [61]:
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(lv_assocs)

,Phecode,Trait description,Sample size,Cases,FDR
